# Retrieval ablation su ~2.000 film: lingua, sparse/dense, nuovi embedding e reranking

Questo notebook riprende il protocollo del precedente ablation study ma **non ripete l'ablation A/B/C/D**: la rappresentazione documentale viene congelata alla configurazione completa `D_full_enriched`, usando `tmdb_title` al posto di `canonical_title` (v2).

L'obiettivo è rispondere ai punti emersi dal confronto con il referente:

1. ripetere le baseline attuali con query in inglese, allineando lingua della query e corpus;
2. separare BGE-M3 **dense** e **learned sparse** e testarli anche in modalità ibrida;
3. confrontare BGE-M3 con `Qwen3-Embedding-0.6B` e `gte-multilingual-base`;
4. provare RRF tra BGE sparse e gli altri dense retriever;
5. solo dopo il confronto del first-stage retrieval, provare più reranker sui sistemi finalisti.

Il notebook usa il corpus arricchito di circa 2.000 film come **proxy benchmark**. La scelta finale dovrà comunque essere rivalutata sul dominio educativo.


## Protocollo sperimentale

**Rappresentazione fissa**

`tmdb_title + genres + overview + tagline + keywords + user tags + directors + writers + composers + cast + characters + collection + production companies`.

**First-stage retrieval**

- BM25 (baseline lessicale);
- BGE-M3 dense;
- BGE-M3 sparse;
- BM25 + BGE-M3 dense (RRF, configurazione già usata);
- BGE-M3 dense + sparse (RRF);
- BM25 + BGE-M3 dense + sparse (RRF);
- Qwen3-Embedding-0.6B dense;
- GTE multilingual base dense;
- Qwen dense + BGE-M3 sparse (RRF);
- GTE dense + BGE-M3 sparse (RRF).

La valutazione principale usa query **in inglese**. Un sottoinsieme IT/EN appaiato misura separatamente l'effetto del mismatch linguistico.

**Valutazione a due profondità**

Per non trasformare il relevance pooling in migliaia di giudizi non necessari:

1. **screening**: pool dell'unione dei top-10 di tutti i sistemi e confronto `NDCG@10`, `Precision@10`, `Recall@10` sui qrels pooled;
2. **deep evaluation dei finalisti**: dopo lo screening si estende il pool ai top-30 dei soli sistemi finalisti; su questo pool si misurano anche `Recall@20` e `Recall@30` e si provano i reranker.

I reranker ricevono candidate pool 10/20/30 e producono sempre top-10 finali.

## 0. Dipendenze

Dalla root del progetto installare l'extra sperimentale:

```bash
python -m pip install -e ".[experiments]"
```


In [69]:
# permette di ricaricare automaticamente i moduli Python quando vengono modificati, senza dover riavviare il kernel

%load_ext autoreload
%autoreload 2

In [96]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import pickle
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")


def locate_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "movie_rag").is_dir():
            return candidate
    raise RuntimeError("Non trovo src/movie_rag: avviare il notebook dalla cartella del progetto.")


PROJECT_ROOT = locate_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from movie_rag import (
    BASELINE_V2,
    BGEV2M3Reranker,
    build_annotation_extension,
    catalog_fingerprint,
    evaluate_results,
    prepare_catalog,
    rerank_candidates,
)
from movie_rag.filters import dataframe_filter_mask
from movie_rag.experimental_retrieval_v2 import (
    BGEM3MultiEncoder,
    BM25Index,
    DenseModelSpec,
    SentenceTransformerCrossEncoderReranker,
    SentenceTransformerDenseEncoder,
    dense_scores,
    retrieve_from_scores,
    sparse_scores,
    weighted_rrf,
    
    build_unique_rerank_candidates,
    materialize_reranked_runs,
    score_reranker_candidates_incremental,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 100)

print("Project root:", PROJECT_ROOT)
for package in ("pandas", "numpy", "FlagEmbedding", "sentence-transformers", "transformers", "rank-bm25"):
    try:
        print(f"{package}: {importlib_metadata.version(package)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{package}: non installato")


Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto
pandas: 3.0.5
numpy: 2.5.1
FlagEmbedding: 1.4.2
sentence-transformers: 5.7.0
transformers: 4.57.6
rank-bm25: 0.2.2


## 1. Configurazione centrale

La configurazione `BASELINE_V2` usa `tmdb_title`, ma **non sostituisce `BASELINE`**: la pipeline Qdrant/MCP esistente rimane pinning alla v1 finché questo nuovo confronto non è concluso.


In [98]:
DATASET_PATH = PROJECT_ROOT / "datasets" / "processed" / "dataset_enriched_movielens_tmdb.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "retrieval_ablation_2000_v1"
CACHE_DIR = ARTIFACT_DIR / "cache"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ID_COLUMN = BASELINE_V2.id_column
TITLE_COLUMN = BASELINE_V2.title_column
DOCUMENT_COLUMN = BASELINE_V2.document_column
EXPERIMENT_NAME = "D_full_enriched_v2"

TOP_K = 10
SCREEN_POOL_DEPTH = 10
DEEP_POOL_DEPTH = 30
RETRIEVAL_DEPTH = 30       # preserva candidati per deep pooling e rerank @30
CANDIDATE_POOL_SIZES = (10, 20, 30)
RRF_K = 60

# Pesi non ottimizzati: il primo confronto usa contributi uguali.
RRF_WEIGHTS_2 = (0.5, 0.5)
RRF_WEIGHTS_3 = (1/3, 1/3, 1/3)

# Modelli suggeriti dal referente.
QWEN_SPEC = DenseModelSpec(
    name="qwen3_0.6b_dense",
    model_name="Qwen/Qwen3-Embedding-0.6B",
    dimension=1024,
    query_prompt_name="query",
)
GTE_SPEC = DenseModelSpec(
    name="gte_multilingual_base_dense",
    model_name="Alibaba-NLP/gte-multilingual-base",
    dimension=768,
    trust_remote_code=True,
)

FIRST_STAGE_RESULTS_PATH = ARTIFACT_DIR / "first_stage_results.csv"
SCREEN_ANNOTATION_POOL_PATH = ARTIFACT_DIR / "annotation_pool_screen_top10.csv"
DEEP_ANNOTATION_POOL_PATH = ARTIFACT_DIR / "annotation_pool_finalists_top30.csv"
LANGUAGE_CONTROL_POOL_PATH = ARTIFACT_DIR / "annotation_pool_language_control.csv"
RERANK_RESULTS_PATH = ARTIFACT_DIR / "rerank_results.csv"

RERANK_CANDIDATES_PATH = ARTIFACT_DIR / "rerank_candidates_unique.csv"
RERANK_SCORE_DIR = ARTIFACT_DIR / "reranker_scores"
RERANK_SCORE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Corpus e rappresentazione documentale v2

Il fetch richiesto per 2.000 film può produrre qualche elemento in meno (ad esempio TMDb 404). Il notebook usa quindi **tutto il dataset arricchito disponibile**, senza imporre esattamente 2.000 righe.


In [7]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset non trovato: {DATASET_PATH}. Eseguire prima il notebook 02.")

source_df = pd.read_csv(DATASET_PATH)
prepared_df = prepare_catalog(source_df, config=BASELINE_V2)
dataset_fingerprint = catalog_fingerprint(prepared_df, config=BASELINE_V2)

print("Film disponibili:", len(prepared_df))
print("Fingerprint:", dataset_fingerprint)
if len(prepared_df) < 1800:
    print("WARNING: il corpus è sensibilmente più piccolo dei ~2000 film previsti.")

display(prepared_df[[ID_COLUMN, TITLE_COLUMN, DOCUMENT_COLUMN]].head(3))


Film disponibili: 1997
Fingerprint: 685fcb1e1ce0cad8c0ea022d580c991070be50116f1870a8af1bca618f64c575


,movieId,tmdb_title,document__D_full_enriched_v2
0,356,Forrest Gump,"Title: Forrest Gump\ngenres: Comedy, Drama, Romance\nOverview: A man with a low IQ has accomplished great things in his life and been pr..."
1,318,The Shawshank Redemption,"Title: The Shawshank Redemption\ngenres: Drama, Crime\nOverview: Imprisoned in the 1940s for the double murder of his wife and her lover..."
2,296,Pulp Fiction,"Title: Pulp Fiction\ngenres: Thriller, Crime, Comedy\nOverview: A burger-loving hit man, his philosophical partner, a drug-addled gangst..."


## 3. Query set principale e controllo linguistico

Il benchmark principale viene eseguito in inglese. Le query italiane sotto sono equivalenti a cinque query inglesi e servono soltanto a quantificare l'effetto del mismatch linguistico senza introdurre una seconda ground truth.


In [21]:
BENCHMARK_QUERIES = pd.DataFrame([
    {"query_id":"Q01", "category":"semantic_theme", "query":"a melancholic film about memory, identity and time", "filters":{}},
    {"query_id":"Q02", "category":"mood", "query":"an emotional, optimistic and moving film", "filters":{}},
    {"query_id":"Q03", "category":"dystopian_theme", "query":"a dystopian film about surveillance, social control and individual freedom", "filters":{}},
    {"query_id":"Q04", "category":"underdog_theme", "query":"a film about an underdog overcoming failure through perseverance, teamwork and personal growth", "filters":{}},
    {"query_id":"Q05", "category":"collection", "query":"a film belonging to a famous saga or collection", "filters":{}},
    {"query_id":"Q06", "category":"mixed_constraints", "query":"a reflective science fiction film under two hours", "filters":{"max_runtime":120, "include_genres":["Science Fiction", "Sci-Fi"]}},
    {"query_id":"Q07", "category":"exclusion", "query":"a tense film that is not horror", "filters":{"exclude_genres":["Horror"]}},
    {"query_id":"Q08", "category":"year_constraint", "query":"a modern drama about family relationships", "filters":{"min_year":2000}},
    {"query_id":"Q09", "category":"country_language", "query":"a chill and sweet French film", "filters":{"countries":["FR"]}},
    # {"query_id":"Q09", "category":"country_language", "query":"an Italian film about family and social change", "filters":{"countries":["IT"], "languages":["it"]}},
    {"query_id":"Q10", "category":"crime_theme", "query":"a tense crime film about a heist, loyalty and betrayal", "filters":{}},
    {"query_id":"Q11", "category":"animation_theme", "query":"an animated film about friendship, growing up and belonging", "filters":{"include_genres":["Animation"]}},
    {"query_id":"Q12", "category":"war_theme", "query":"a war film about sacrifice, duty and the human cost of combat", "filters":{}},
    {"query_id":"Q13", "category":"ai_theme", "query":"a science fiction film about artificial intelligence and what it means to be human", "filters":{}},
    {"query_id":"Q14", "category":"justice_theme", "query":"a courtroom drama about justice, ethics and moral conflict", "filters":{}},
    {"query_id":"Q15", "category":"romance_theme", "query":"a witty romantic comedy about two people gradually falling in love", "filters":{"include_genres":["Romance", "Comedy"]}},
    {"query_id":"Q16", "category":"psychological_theme", "query":"a psychological thriller with unreliable memory and a fragmented sense of reality", "filters":{}},
    {"query_id":"Q17", "category":"space_theme", "query":"a science fiction film about space exploration, isolation and survival", "filters":{"include_genres":["Science Fiction", "Sci-Fi"]}},
    {"query_id":"Q18", "category":"coming_of_age", "query":"a coming-of-age film about school, friendship and finding your identity", "filters":{}},
    {"query_id":"Q19", "category":"fantasy_theme", "query":"a fantasy adventure involving a dangerous quest, magic and friendship", "filters":{"include_genres":["Fantasy", "Adventure"]}},
    {"query_id":"Q20", "category":"music_theme", "query":"a film about music, artistic ambition and the personal cost of success", "filters":{}},
])

LANGUAGE_CONTROL_QUERIES = pd.DataFrame([
    {"query_id":"L01_IT", "source_query_id":"Q01", "query":"un film malinconico sul rapporto tra memoria, identità e tempo", "filters":{}},
    {"query_id":"L03_IT", "source_query_id":"Q03", "query":"un film con Tom Hanks", "filters":{}},
    {"query_id":"L04_IT", "source_query_id":"Q04", "query":"un film diretto da Christopher Nolan", "filters":{}},
    {"query_id":"L06_IT", "source_query_id":"Q06", "query":"un film di fantascienza riflessivo sotto le due ore", "filters":{"max_runtime":120, "include_genres":["Science Fiction", "Sci-Fi"]}},
    {"query_id":"L07_IT", "source_query_id":"Q07", "query":"un film teso ma non horror", "filters":{"exclude_genres":["Horror"]}},
])

display(BENCHMARK_QUERIES)
display(LANGUAGE_CONTROL_QUERIES)


,query_id,category,query,filters
0,Q01,semantic_theme,"a melancholic film about memory, identity and time",{}
1,Q02,mood,"an emotional, optimistic and moving film",{}
2,Q03,dystopian_theme,"a dystopian film about surveillance, social control and individual freedom",{}
3,Q04,underdog_theme,"a film about an underdog overcoming failure through perseverance, teamwork and personal growth",{}
4,Q05,collection,a film belonging to a famous saga or collection,{}
5,Q06,mixed_constraints,a reflective science fiction film under two hours,"{'max_runtime': 120, 'include_genres': ['Science Fiction', 'Sci-Fi']}"
6,Q07,exclusion,a tense film that is not horror,{'exclude_genres': ['Horror']}
7,Q08,year_constraint,a modern drama about family relationships,{'min_year': 2000}
8,Q09,country_language,a chill and sweet French film,{'countries': ['FR']}
9,Q10,crime_theme,"a tense crime film about a heist, loyalty and betrayal",{}


,query_id,source_query_id,query,filters
0,L01_IT,Q01,"un film malinconico sul rapporto tra memoria, identità e tempo",{}
1,L03_IT,Q03,un film con Tom Hanks,{}
2,L04_IT,Q04,un film diretto da Christopher Nolan,{}
3,L06_IT,Q06,un film di fantascienza riflessivo sotto le due ore,"{'max_runtime': 120, 'include_genres': ['Science Fiction', 'Sci-Fi']}"
4,L07_IT,Q07,un film teso ma non horror,{'exclude_genres': ['Horror']}


### 3.1. Preflight dei filtri

Prima di eseguire i modelli controlliamo soltanto che i vincoli strutturati non svuotino accidentalmente lo spazio di ricerca. Questo controllo **non** stabilisce che esistano documenti semanticamente rilevanti: serve solo a individuare query impossibili a causa dei filtri.

In [12]:
filter_audit = []
for row in BENCHMARK_QUERIES.to_dict("records"):
    mask = dataframe_filter_mask(prepared_df, row.get("filters") or {})
    eligible = int(mask.sum())
    filter_audit.append({
        "query_id": row["query_id"],
        "category": row["category"],
        "eligible_after_filters": eligible,
        "warning": eligible < TOP_K,
    })

filter_audit = pd.DataFrame(filter_audit)
display(filter_audit)
if filter_audit["warning"].any():
    print("WARNING: alcune query hanno meno di TOP_K film eleggibili; rivedere i filtri prima del benchmark.")

,query_id,category,eligible_after_filters,warning
0,Q01,semantic_theme,1997,False
1,Q02,mood,1997,False
2,Q03,named_entity_cast,1997,False
3,Q04,named_entity_director,1997,False
4,Q05,collection,1997,False
5,Q06,mixed_constraints,247,False
6,Q07,exclusion,1834,False
7,Q08,year_constraint,820,False
8,Q09,country_language,115,False
9,Q10,crime_theme,1997,False


## 4. Cache delle rappresentazioni

Per rendere il notebook rieseguibile, document vectors e query vectors vengono memorizzati localmente. La cache è invalidata automaticamente dal fingerprint del catalogo e dal nome del modello.


In [14]:
def safe_name(value: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in value)


def dense_cache_path(model_name: str) -> Path:
    return CACHE_DIR / f"{safe_name(model_name)}__{dataset_fingerprint[:12]}.npz"


def save_dense_cache(path: Path, vectors: np.ndarray) -> None:
    np.savez_compressed(path, vectors=np.asarray(vectors, dtype=np.float32))


def load_dense_cache(path: Path, expected_rows: int, expected_dim: int) -> np.ndarray | None:
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as data:
        vectors = np.asarray(data["vectors"], dtype=np.float32)
    if vectors.shape != (expected_rows, expected_dim):
        return None
    return vectors


def bge_sparse_cache_path() -> Path:
    return CACHE_DIR / f"bge_m3_sparse__{dataset_fingerprint[:12]}.pkl"


## 5. BM25 e BGE-M3 dense + sparse

BGE-M3 viene codificato una sola volta ottenendo contemporaneamente `dense_vecs` e `lexical_weights`. La componente sparse è quindi una **learned sparse representation**, distinta dal BM25 classico.


In [15]:
DOCUMENTS = prepared_df[DOCUMENT_COLUMN].astype(str).tolist()
BM25 = BM25Index(DOCUMENTS).build()

BGE_DENSE_CACHE = dense_cache_path("BAAI_bge-m3_dense")
BGE_SPARSE_CACHE = bge_sparse_cache_path()

bge_dense_vectors = load_dense_cache(BGE_DENSE_CACHE, len(prepared_df), 1024)
bge_sparse_documents = None
if BGE_SPARSE_CACHE.exists():
    with BGE_SPARSE_CACHE.open("rb") as handle:
        payload = pickle.load(handle)
    if payload.get("fingerprint") == dataset_fingerprint and len(payload.get("weights", [])) == len(prepared_df):
        bge_sparse_documents = payload["weights"]

bge_encoder = BGEM3MultiEncoder()
if bge_dense_vectors is None or bge_sparse_documents is None:
    started = perf_counter()
    dense_new, sparse_new = bge_encoder.encode_documents(DOCUMENTS)
    print(f"BGE-M3 document encoding: {perf_counter()-started:.1f}s")
    if bge_dense_vectors is None:
        bge_dense_vectors = dense_new
        save_dense_cache(BGE_DENSE_CACHE, bge_dense_vectors)
    if bge_sparse_documents is None:
        bge_sparse_documents = sparse_new
        with BGE_SPARSE_CACHE.open("wb") as handle:
            pickle.dump({"fingerprint": dataset_fingerprint, "weights": bge_sparse_documents}, handle)

# Query representations: benchmark + language controls, per evitare di ricaricare BGE più avanti.
ALL_QUERY_ROWS = pd.concat([
    BENCHMARK_QUERIES[["query_id", "query"]],
    LANGUAGE_CONTROL_QUERIES[["query_id", "query"]],
], ignore_index=True)
BGE_QUERY_REP = {}
for row in ALL_QUERY_ROWS.to_dict("records"):
    dense_q, sparse_q = bge_encoder.encode_query(row["query"])
    BGE_QUERY_REP[row["query_id"]] = {"dense": dense_q, "sparse": sparse_q}

bge_encoder.unload()
print("BGE dense shape:", bge_dense_vectors.shape)
print("BGE sparse docs:", len(bge_sparse_documents))


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 125/125 [00:00<00:00, 385.38it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 125/125 [01:13<00:00,  1.70it/s]


BGE-M3 document encoding: 90.5s


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 23.66it/s]


BGE dense shape: (1997, 1024)
BGE sparse docs: 1997


## 6. Qwen3-Embedding-0.6B e GTE multilingual base

I due modelli vengono caricati **in sequenza** per limitare la memoria GPU. Qwen usa il prompt query raccomandato dal model card; i documenti vengono codificati senza prompt.


In [16]:
def prepare_dense_model(spec: DenseModelSpec, query_rows: pd.DataFrame):
    cache_path = dense_cache_path(spec.name)
    document_vectors = load_dense_cache(cache_path, len(prepared_df), spec.dimension)
    encoder = SentenceTransformerDenseEncoder(spec=spec)
    if document_vectors is None:
        started = perf_counter()
        document_vectors = encoder.encode_documents(DOCUMENTS)
        print(f"{spec.name} document encoding: {perf_counter()-started:.1f}s")
        save_dense_cache(cache_path, document_vectors)
    query_vectors = {
        row["query_id"]: encoder.encode_query(row["query"])
        for row in query_rows.to_dict("records")
    }
    encoder.unload()
    return document_vectors, query_vectors


qwen_dense_vectors, QWEN_QUERY_REP = prepare_dense_model(QWEN_SPEC, ALL_QUERY_ROWS)
gte_dense_vectors, GTE_QUERY_REP = prepare_dense_model(GTE_SPEC, ALL_QUERY_ROWS)

print("Qwen shape:", qwen_dense_vectors.shape)
print("GTE shape:", gte_dense_vectors.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (18 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

gte_multilingual_base_dense document encoding: 234.5s
Qwen shape: (1997, 1024)
GTE shape: (1997, 768)


## 7. First-stage retrieval grid

Ogni sistema restituisce fino a 30 candidati. Gli hybrid usano RRF sui **rank**, quindi non richiedono normalizzazione degli score BM25/dense/sparse.


In [22]:
def result_records(frame: pd.DataFrame, query_row: dict, system: str) -> list[dict]:
    rows = []
    for item in frame.to_dict("records"):
        rows.append({
            "query_id": query_row["query_id"],
            "query": query_row["query"],
            "category": query_row.get("category", "language_control"),
            "experiment": EXPERIMENT_NAME,
            "system": system,
            "stage": "first_stage",
            "rank": int(item["rank"]),
            ID_COLUMN: item[ID_COLUMN],
            TITLE_COLUMN: item.get(TITLE_COLUMN),
            "score": float(item["score"]),
        })
    return rows


def retrieve_base_frames(query_row: dict) -> dict[str, pd.DataFrame]:
    qid = query_row["query_id"]
    query = query_row["query"]
    filters = query_row.get("filters") or {}

    bm25 = retrieve_from_scores(prepared_df, BM25.scores(query), RETRIEVAL_DEPTH, filters,
                                ID_COLUMN, TITLE_COLUMN, DOCUMENT_COLUMN)
    bge_dense = retrieve_from_scores(
        prepared_df,
        dense_scores(BGE_QUERY_REP[qid]["dense"], bge_dense_vectors),
        RETRIEVAL_DEPTH, filters, ID_COLUMN, TITLE_COLUMN, DOCUMENT_COLUMN,
    )
    bge_sparse = retrieve_from_scores(
        prepared_df,
        sparse_scores(BGE_QUERY_REP[qid]["sparse"], bge_sparse_documents),
        RETRIEVAL_DEPTH, filters, ID_COLUMN, TITLE_COLUMN, DOCUMENT_COLUMN,
    )
    qwen = retrieve_from_scores(
        prepared_df,
        dense_scores(QWEN_QUERY_REP[qid], qwen_dense_vectors),
        RETRIEVAL_DEPTH, filters, ID_COLUMN, TITLE_COLUMN, DOCUMENT_COLUMN,
    )
    gte = retrieve_from_scores(
        prepared_df,
        dense_scores(GTE_QUERY_REP[qid], gte_dense_vectors),
        RETRIEVAL_DEPTH, filters, ID_COLUMN, TITLE_COLUMN, DOCUMENT_COLUMN,
    )
    return {
        "bm25": bm25,
        "bge_m3_dense": bge_dense,
        "bge_m3_sparse": bge_sparse,
        "qwen3_0.6b_dense": qwen,
        "gte_multilingual_base_dense": gte,
    }


def fuse(frames: dict[str, pd.DataFrame], names: list[str], weights: list[float]) -> pd.DataFrame:
    return weighted_rrf(
        {name: frames[name] for name in names},
        weights=dict(zip(names, weights)),
        rrf_k=RRF_K,
        top_k=RETRIEVAL_DEPTH,
        id_column=ID_COLUMN,
    )


def run_first_stage(query_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for query_row in query_df.to_dict("records"):
        frames = retrieve_base_frames(query_row)
        frames["bm25+bge_m3_dense_rrf"] = fuse(
            frames, ["bm25", "bge_m3_dense"], list(RRF_WEIGHTS_2)
        )
        frames["bge_m3_dense+sparse_rrf"] = fuse(
            frames, ["bge_m3_dense", "bge_m3_sparse"], list(RRF_WEIGHTS_2)
        )
        frames["bm25+bge_m3_dense+sparse_rrf"] = fuse(
            frames, ["bm25", "bge_m3_dense", "bge_m3_sparse"], list(RRF_WEIGHTS_3)
        )
        frames["qwen3_dense+bge_m3_sparse_rrf"] = fuse(
            frames, ["qwen3_0.6b_dense", "bge_m3_sparse"], list(RRF_WEIGHTS_2)
        )
        frames["gte_dense+bge_m3_sparse_rrf"] = fuse(
            frames, ["gte_multilingual_base_dense", "bge_m3_sparse"], list(RRF_WEIGHTS_2)
        )
        for system, frame in frames.items():
            records.extend(result_records(frame, query_row, system))
    return pd.DataFrame(records)


first_stage_results = run_first_stage(BENCHMARK_QUERIES)
first_stage_results.to_csv(FIRST_STAGE_RESULTS_PATH, index=False)
print("Righe first-stage:", len(first_stage_results))
display(first_stage_results.head(20))


Righe first-stage: 6000


,query_id,query,category,experiment,system,stage,rank,movieId,tmdb_title,score
0,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,1,1748,Dark City,13.418836
1,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,2,5459,Men in Black II,12.599925
2,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,3,2916,Total Recall,12.545115
3,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,4,2908,Boys Don't Cry,12.143422
4,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,5,2973,Crimes and Misdemeanors,11.917349
5,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,6,2657,The Rocky Horror Picture Show,11.867302
6,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,7,110553,The Amazing Spider-Man 2,11.721454
7,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,8,7361,Eternal Sunshine of the Spotless Mind,11.392774
8,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,9,7163,Paycheck,11.346609
9,Q01,"a melancholic film about memory, identity and time",semantic_theme,D_full_enriched_v2,bm25,first_stage,10,1047,The Long Kiss Goodnight,11.328011


## 8. Screening pool: top-10 di tutti i retriever

Il vecchio pool sui 50 film non è riutilizzabile sul nuovo corpus. Per la prima selezione costruiamo un nuovo relevance pool come unione dei **top-10 di tutti i first-stage systems**. Ogni coppia query-film viene giudicata una sola volta con `0/1/2`.

Questo pool serve a scegliere i sistemi che meritano una valutazione più profonda. Non viene ancora richiesto di annotare i top-20/30 di tutte le configurazioni.

In [23]:
POOL_COLUMNS = ["query_id", "query", ID_COLUMN, TITLE_COLUMN, "retrieved_by", "best_rank", "relevance", "notes"]

if SCREEN_ANNOTATION_POOL_PATH.exists():
    screen_pool = pd.read_csv(SCREEN_ANNOTATION_POOL_PATH)
else:
    screen_pool = pd.DataFrame(columns=POOL_COLUMNS)

screen_results = first_stage_results[first_stage_results["rank"] <= SCREEN_POOL_DEPTH].copy()
screen_extension = build_annotation_extension(
    results=screen_results,
    annotated_pool=screen_pool,
    id_column=ID_COLUMN,
    title_column=TITLE_COLUMN,
)

if not screen_extension.empty:
    screen_pool = pd.concat([screen_pool, screen_extension], ignore_index=True)
    screen_pool = screen_pool.drop_duplicates(["query_id", ID_COLUMN], keep="first")
    screen_pool = screen_pool.sort_values(["query_id", "best_rank", ID_COLUMN], kind="stable")
    screen_pool.to_csv(SCREEN_ANNOTATION_POOL_PATH, index=False)

print("Screening pool totale:", len(screen_pool))
print("Nuovi giudizi richiesti:", len(screen_extension))
print("Da annotare in:", SCREEN_ANNOTATION_POOL_PATH)
display(screen_extension.head(30))

Screening pool totale: 712
Nuovi giudizi richiesti: 712
Da annotare in: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/artifacts/retrieval_ablation_2000_v1/annotation_pool_screen_top10.csv


,query_id,query,movieId,tmdb_title,retrieved_by,best_rank,relevance,notes
0,Q01,"a melancholic film about memory, identity and time",1748,Dark City,bge_m3_dense+sparse_rrf@3 | bge_m3_dense@1 | bm25+bge_m3_dense+sparse_rrf@1 | bm25+bge_m3_dense_rrf@1 | bm25@1 | gte_dense+bge_m3_sparse...,1,<NA>,
1,Q01,"a melancholic film about memory, identity and time",4226,Memento,bge_m3_dense+sparse_rrf@7 | bge_m3_dense@3 | bm25+bge_m3_dense_rrf@9 | gte_dense+bge_m3_sparse_rrf@6 | gte_multilingual_base_dense@1 | q...,1,<NA>,
2,Q01,"a melancholic film about memory, identity and time",4878,Donnie Darko,bge_m3_dense+sparse_rrf@1 | bge_m3_dense@4 | bge_m3_sparse@3 | bm25+bge_m3_dense+sparse_rrf@4 | gte_dense+bge_m3_sparse_rrf@2 | gte_mult...,1,<NA>,
3,Q01,"a melancholic film about memory, identity and time",7361,Eternal Sunshine of the Spotless Mind,bge_m3_dense+sparse_rrf@4 | bge_m3_sparse@2 | bm25+bge_m3_dense+sparse_rrf@2 | bm25+bge_m3_dense_rrf@4 | bm25@8 | gte_dense+bge_m3_spars...,1,<NA>,
4,Q01,"a melancholic film about memory, identity and time",34319,The Island,bge_m3_dense+sparse_rrf@6 | bge_m3_sparse@1 | bm25+bge_m3_dense+sparse_rrf@7 | gte_dense+bge_m3_sparse_rrf@5 | qwen3_dense+bge_m3_sparse...,1,<NA>,
5,Q01,"a melancholic film about memory, identity and time",5459,Men in Black II,bm25+bge_m3_dense+sparse_rrf@8 | bm25+bge_m3_dense_rrf@6 | bm25@2,2,<NA>,
6,Q01,"a melancholic film about memory, identity and time",6323,Identity,bge_m3_dense+sparse_rrf@2 | bge_m3_dense@2 | bm25+bge_m3_dense+sparse_rrf@9 | bm25+bge_m3_dense_rrf@7,2,<NA>,
7,Q01,"a melancholic film about memory, identity and time",7254,The Butterfly Effect,bge_m3_dense@7 | bm25+bge_m3_dense+sparse_rrf@10 | bm25+bge_m3_dense_rrf@2 | gte_multilingual_base_dense@7 | qwen3_0.6b_dense@5,2,<NA>,
8,Q01,"a melancholic film about memory, identity and time",1047,The Long Kiss Goodnight,bge_m3_dense+sparse_rrf@5 | bm25+bge_m3_dense+sparse_rrf@3 | bm25+bge_m3_dense_rrf@3 | bm25@10 | gte_dense+bge_m3_sparse_rrf@3 | qwen3_d...,3,<NA>,
9,Q01,"a melancholic film about memory, identity and time",2916,Total Recall,bge_m3_sparse@9 | bm25+bge_m3_dense+sparse_rrf@6 | bm25+bge_m3_dense_rrf@8 | bm25@3,3,<NA>,


In [24]:
input_path = Path(
    "/Users/danieleraveglia/Documents/uniMi/TESI/progetto/"
    "artifacts/retrieval_ablation_2000_v1/annotation_pool_screen_top10.csv"
)
output_path = input_path.with_name("datiFiltrati.csv")

columns = ["query_id", "query", "tmdb_title"]
filtered_df = pd.read_csv(input_path, usecols=columns)
filtered_df.to_csv(output_path, index=False)

print(f"File salvato in: {output_path}")
display(filtered_df.head())

File salvato in: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/artifacts/retrieval_ablation_2000_v1/datiFiltrati.csv


,query_id,query,tmdb_title
0,Q01,"a melancholic film about memory, identity and time",Dark City
1,Q01,"a melancholic film about memory, identity and time",Memento
2,Q01,"a melancholic film about memory, identity and time",Donnie Darko
3,Q01,"a melancholic film about memory, identity and time",Eternal Sunshine of the Spotless Mind
4,Q01,"a melancholic film about memory, identity and time",The Island


In [26]:
input_path = Path(
    "/Users/danieleraveglia/Documents/uniMi/TESI/progetto/"
    "artifacts/retrieval_ablation_2000_v1/manual_annotations_pool_screen-top10.csv"
)

target_path = SCREEN_ANNOTATION_POOL_PATH
target_df = pd.read_csv(target_path)
manual_df = pd.read_csv(input_path)

# Normalizzazione minima
for df in (target_df, manual_df):
    for col in ["query_id", "tmdb_title"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# Controllo colonne richieste
required = ["query_id", "tmdb_title", "relevance"]
missing_manual = [c for c in required if c not in manual_df.columns]
if missing_manual:
    raise ValueError(f"Il file input manca delle colonne: {missing_manual}")

# Se il file manuale ha lo stesso numero di righe e lo stesso ordine, usa l'assegnazione diretta
if len(target_df) == len(manual_df):
    target_df["relevance"] = pd.to_numeric(manual_df["relevance"], errors="coerce")
    target_df.to_csv(target_path, index=False)
    print("relevance aggiornato per ordine delle righe.")
else:
    # Fallback robusto: match su query_id + tmdb_title
    merged = target_df.merge(
        manual_df[["query_id", "tmdb_title", "relevance"]].rename(
            columns={"relevance": "relevance_manual"}
        ),
        on=["query_id", "tmdb_title"],
        how="left",
    )

    # Mantiene il valore già presente nel target solo se il manuale non lo ha
    merged["relevance"] = (
        pd.to_numeric(merged["relevance_manual"], errors="coerce")
        .combine_first(pd.to_numeric(merged["relevance"], errors="coerce"))
    )

    merged = merged.drop(columns=["relevance_manual"], errors="ignore")
    merged.to_csv(target_path, index=False)
    print("relevance aggiornato con match query_id + tmdb_title.")

# Controllo finale
updated = pd.read_csv(target_path)
print("Colonne:", updated.columns.tolist())
print("relevance nulli:", int(updated["relevance"].isna().sum()))
display(updated[["query_id", "tmdb_title", "relevance"]].head(10))



relevance aggiornato per ordine delle righe.
Colonne: ['query_id', 'query', 'movieId', 'tmdb_title', 'retrieved_by', 'best_rank', 'relevance', 'notes']
relevance nulli: 0


,query_id,tmdb_title,relevance
0,Q01,Dark City,2
1,Q01,Memento,2
2,Q01,Donnie Darko,2
3,Q01,Eternal Sunshine of the Spotless Mind,2
4,Q01,The Island,1
5,Q01,Men in Black II,0
6,Q01,Identity,1
7,Q01,The Butterfly Effect,2
8,Q01,The Long Kiss Goodnight,1
9,Q01,Total Recall,1


## 9. Metriche di screening @10

Questa fase confronta tutti i sistemi sulla lista effettivamente mostrata. `Recall@10` è una recall **rispetto al pool giudicato**, non una recall esaustiva sull'intero corpus. La candidate recall più profonda viene calcolata soltanto dopo aver scelto i finalisti.

In [27]:
screen_pool = pd.read_csv(SCREEN_ANNOTATION_POOL_PATH)
screen_missing = int(screen_pool["relevance"].isna().sum())
print("Giudizi screening mancanti:", screen_missing)

if screen_missing == 0 and len(screen_pool) > 0:
    screen_metrics = evaluate_results(
        first_stage_results[first_stage_results["rank"] <= TOP_K],
        screen_pool,
        k=TOP_K,
        id_column=ID_COLUMN,
        expected_query_ids=BENCHMARK_QUERIES["query_id"].tolist(),
        strict_judgments=True,
    )
    screen_summary = (
        screen_metrics.groupby("system")[["ndcg@10", "recall@10", "precision@10"]]
        .mean()
        .sort_values("ndcg@10", ascending=False)
    )
    display(screen_summary)
else:
    print("Completare prima i giudizi 0/1/2 dello screening pool.")

Giudizi screening mancanti: 0


,ndcg@10,recall@10,precision@10
system,,,
gte_multilingual_base_dense,0.794889,0.333658,0.900
qwen3_0.6b_dense,0.759249,0.332600,0.890
bm25+bge_m3_dense_rrf,0.707442,0.331887,0.835
gte_dense+bge_m3_sparse_rrf,0.697888,0.301819,0.820
bge_m3_dense,0.681320,0.299703,0.810
qwen3_dense+bge_m3_sparse_rrf,0.667348,0.297223,0.805
bm25+bge_m3_dense+sparse_rrf,0.609574,0.288338,0.760
bge_m3_dense+sparse_rrf,0.607949,0.284825,0.775
bm25,0.535350,0.291141,0.705


### risultati ottenuti:

| sistema | NDCG@10 | recall@10 | precision@10 
|---------|------|--------|------|
| `gte_multilingual_base_dense` |0.794889	|0.333658	|0.900|
| `qwen3_0.6b_dense` |0.759249	|0.332600	|0.890|
| `bm25+bge_m3_dense_rrf` |0.707442	|0.331887	|0.835|
| `gte_dense+bge_m3_sparse_rrf` |0.697888	|0.301819	|0.820|


## 10. Controllo appaiato IT/EN

Questo test isola l'ipotesi emersa dal confronto con il referente: BM25 e l'hybrid originario dovrebbero beneficiare dell'allineamento linguistico tra query e documenti. Aggiungiamo anche BGE-M3 sparse come controllo.

Il pool linguistico viene costruito sull'**unione dei risultati EN e IT equivalenti**. Il giudizio viene assegnato una sola volta alla richiesta semantica sottostante e poi riutilizzato per entrambe le lingue: in questo modo un risultato recuperato soltanto dalla query italiana non viene implicitamente trattato come irrilevante.

In [29]:
# def run_language_control() -> pd.DataFrame:
#     records = []
#     english_lookup = BENCHMARK_QUERIES.set_index("query_id")
#     control_rows = []
#     for row in LANGUAGE_CONTROL_QUERIES.to_dict("records"):
#         english = english_lookup.loc[row["source_query_id"]].to_dict()
#         english["query_id"] = f"{row['source_query_id']}_EN_CTRL"
#         english["source_query_id"] = row["source_query_id"]
#         control_rows.append(english)
#         control_rows.append(row)

#     for query_row in control_rows:
#         source_id = query_row.get("source_query_id") or query_row["query_id"].replace("_EN_CTRL", "")
#         actual_query_row = dict(query_row)
#         output_query_id = actual_query_row["query_id"]
#         # Le rappresentazioni della query EN coincidono con quelle già calcolate per la benchmark query.
#         if output_query_id.endswith("_EN_CTRL"):
#             actual_query_row["query_id"] = source_id
#         frames = retrieve_base_frames(actual_query_row)
#         hybrid = fuse(frames, ["bm25", "bge_m3_dense"], list(RRF_WEIGHTS_2))
#         selected = {
#             "bm25": frames["bm25"],
#             "bge_m3_dense": frames["bge_m3_dense"],
#             "bge_m3_sparse": frames["bge_m3_sparse"],
#             "bm25+bge_m3_dense_rrf": hybrid,
#         }
#         for system, frame in selected.items():
#             out_row = dict(query_row)
#             out_row["query_id"] = output_query_id
#             records.extend(result_records(frame, out_row, system))
#     return pd.DataFrame(records)


# language_results = run_language_control()
# print("Language-control rows:", len(language_results))
# display(language_results.head(20))

# # Per l'annotazione collassiamo EN e IT sulla stessa semantic request/source query.
# control_to_source = {}
# for row in LANGUAGE_CONTROL_QUERIES.to_dict("records"):
#     control_to_source[row["query_id"]] = row["source_query_id"]
#     control_to_source[f"{row['source_query_id']}_EN_CTRL"] = row["source_query_id"]

# language_pool_results = language_results[language_results["rank"] <= TOP_K].copy()
# language_pool_results["query_id"] = language_pool_results["query_id"].map(control_to_source)
# english_text = BENCHMARK_QUERIES.set_index("query_id")["query"].to_dict()
# language_pool_results["query"] = language_pool_results["query_id"].map(english_text)

# if LANGUAGE_CONTROL_POOL_PATH.exists():
#     language_pool = pd.read_csv(LANGUAGE_CONTROL_POOL_PATH)
# else:
#     language_pool = pd.DataFrame(columns=POOL_COLUMNS)

# language_extension = build_annotation_extension(
#     language_pool_results,
#     language_pool,
#     id_column=ID_COLUMN,
#     title_column=TITLE_COLUMN,
# )
# if not language_extension.empty:
#     language_pool = pd.concat([language_pool, language_extension], ignore_index=True)
#     language_pool = language_pool.drop_duplicates(["query_id", ID_COLUMN], keep="first")
#     language_pool.to_csv(LANGUAGE_CONTROL_POOL_PATH, index=False)

# print("Language pool totale:", len(language_pool))
# print("Nuovi giudizi language-control:", len(language_extension))
# print("Da annotare in:", LANGUAGE_CONTROL_POOL_PATH)

In [ ]:
# def expand_language_qrels(source_qrels: pd.DataFrame) -> pd.DataFrame:
#     chunks = []
#     for row in LANGUAGE_CONTROL_QUERIES.to_dict("records"):
#         source_id = row["source_query_id"]
#         source = source_qrels[source_qrels["query_id"].eq(source_id)].copy()
#         if source.empty:
#             continue
#         en = source.copy(); en["query_id"] = f"{source_id}_EN_CTRL"
#         it = source.copy(); it["query_id"] = row["query_id"]
#         chunks.extend([en, it])
#     return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()


# language_pool = pd.read_csv(LANGUAGE_CONTROL_POOL_PATH)
# language_missing = int(language_pool["relevance"].isna().sum())
# print("Giudizi language-control mancanti:", language_missing)

# if language_missing == 0 and len(language_pool) > 0:
#     lang_qrels = expand_language_qrels(language_pool)
#     lang_metrics = evaluate_results(
#         language_results[language_results["rank"] <= TOP_K],
#         lang_qrels,
#         k=TOP_K,
#         id_column=ID_COLUMN,
#         strict_judgments=True,
#     )
#     lang_metrics["language"] = np.where(lang_metrics["query_id"].str.contains("_IT"), "it", "en")
#     display(
#         lang_metrics.groupby(["system", "language"])[["ndcg@10", "recall@10", "precision@10"]]
#         .mean()
#     )
# else:
#     print("Completare prima il pool linguistico.")

## 11. Deep pooling dei first-stage finalisti

Dopo lo screening si selezionano pochi sistemi che meritano una valutazione più approfondita. La lista seguente è soltanto un **default iniziale**: va aggiornata guardando i risultati dello screening.

Per i finalisti estendiamo i qrels fino ai **top-30**. Questo consente di confrontare `Recall@20`/`Recall@30` e, soprattutto, garantisce che qualunque documento possa essere portato dal reranker nei top-10 partendo da pool @10/@20/@30 sia già stato giudicato.

In [45]:
FINALIST_SYSTEMS = [
    "gte_multilingual_base_dense",
    "qwen3_0.6b_dense",
    "bm25+bge_m3_dense_rrf",
    "gte_dense+bge_m3_sparse_rrf",
]

# Il deep pool parte dai giudizi di screening, ma viene salvato separatamente.
if DEEP_ANNOTATION_POOL_PATH.exists():
    deep_pool = pd.read_csv(DEEP_ANNOTATION_POOL_PATH)
else:
    deep_pool = pd.read_csv(SCREEN_ANNOTATION_POOL_PATH).copy()

deep_results = first_stage_results[
    first_stage_results["system"].isin(FINALIST_SYSTEMS)
    & first_stage_results["rank"].le(DEEP_POOL_DEPTH)
].copy()

deep_extension = build_annotation_extension(
    deep_results,
    deep_pool,
    id_column=ID_COLUMN,
    title_column=TITLE_COLUMN,
)
if not deep_extension.empty:
    deep_pool = pd.concat([deep_pool, deep_extension], ignore_index=True)
    deep_pool = deep_pool.drop_duplicates(["query_id", ID_COLUMN], keep="first")
    deep_pool = deep_pool.sort_values(["query_id", "best_rank", ID_COLUMN], kind="stable")
    deep_pool.to_csv(DEEP_ANNOTATION_POOL_PATH, index=False)

print("Deep pool totale:", len(deep_pool))
print("Nuovi giudizi richiesti dai finalisti:", len(deep_extension))
print("Da annotare in:", DEEP_ANNOTATION_POOL_PATH)
display(deep_extension.head(30))

Deep pool totale: 1358
Nuovi giudizi richiesti dai finalisti: 646
Da annotare in: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/artifacts/retrieval_ablation_2000_v1/annotation_pool_finalists_top30.csv


,query_id,query,movieId,tmdb_title,retrieved_by,best_rank,relevance,notes
0,Q01,"a melancholic film about memory, identity and time",5528,One Hour Photo,gte_dense+bge_m3_sparse_rrf@22 | gte_multilingual_base_dense@11,11,<NA>,
1,Q01,"a melancholic film about memory, identity and time",104374,About Time,bm25+bge_m3_dense_rrf@27 | gte_multilingual_base_dense@17 | qwen3_0.6b_dense@11,11,<NA>,
2,Q01,"a melancholic film about memory, identity and time",1183,The English Patient,qwen3_0.6b_dense@12,12,<NA>,
3,Q01,"a melancholic film about memory, identity and time",3083,All About My Mother,gte_dense+bge_m3_sparse_rrf@23 | gte_multilingual_base_dense@12,12,<NA>,
4,Q01,"a melancholic film about memory, identity and time",5418,The Bourne Identity,gte_dense+bge_m3_sparse_rrf@24 | gte_multilingual_base_dense@13 | qwen3_0.6b_dense@24,13,<NA>,
5,Q01,"a melancholic film about memory, identity and time",3503,Solaris,bm25+bge_m3_dense_rrf@29 | gte_dense+bge_m3_sparse_rrf@28 | gte_multilingual_base_dense@15,15,<NA>,
6,Q01,"a melancholic film about memory, identity and time",86882,Midnight in Paris,qwen3_0.6b_dense@17,17,<NA>,
7,Q01,"a melancholic film about memory, identity and time",1956,Ordinary People,gte_multilingual_base_dense@18,18,<NA>,
8,Q01,"a melancholic film about memory, identity and time",2496,Blast from the Past,qwen3_0.6b_dense@18,18,<NA>,
9,Q01,"a melancholic film about memory, identity and time",3269,Forever Young,gte_multilingual_base_dense@19 | qwen3_0.6b_dense@27,19,<NA>,


In [47]:
input_path = Path(
    "/Users/danieleraveglia/Documents/uniMi/TESI/progetto/"
    "artifacts/retrieval_ablation_2000_v1/annotation_pool_finalists_top30.csv"
)

output_path = input_path.with_name("datiFiltrati_top30.csv")

columns = ["query_id", "query", "tmdb_title", "relevance"]
filtered_df = pd.read_csv(input_path, usecols=columns)

# Filtra i film senza giudizio e rimuovi la colonna relevance prima di salvare
filtered_df["relevance"] = pd.to_numeric(filtered_df["relevance"], errors="coerce")
filtered_df = filtered_df[~filtered_df["relevance"].notna()].copy()
# filtered_df = filtered_df.drop(columns=["relevance"])

filtered_df.to_csv(output_path, index=False)

print(f"File salvato in: {output_path}")
display(filtered_df.head())

File salvato in: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/artifacts/retrieval_ablation_2000_v1/datiFiltrati_top30.csv


,query_id,query,tmdb_title,relevance
33,Q01,"a melancholic film about memory, identity and time",One Hour Photo,NaN
34,Q01,"a melancholic film about memory, identity and time",About Time,NaN
35,Q01,"a melancholic film about memory, identity and time",The English Patient,NaN
36,Q01,"a melancholic film about memory, identity and time",All About My Mother,NaN
37,Q01,"a melancholic film about memory, identity and time",The Bourne Identity,NaN


In [49]:
input_path = Path(
    "/Users/danieleraveglia/Documents/uniMi/TESI/progetto/"
    "artifacts/retrieval_ablation_2000_v1/manual_DELTA_annotations_pool_screen-top30.csv"
)

target_path = DEEP_ANNOTATION_POOL_PATH
target_df = pd.read_csv(target_path)
manual_df = pd.read_csv(input_path)

# Normalizzazione minima
for df in (target_df, manual_df):
    for col in ["query_id", "tmdb_title"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# Controllo colonne richieste
required = ["query_id", "tmdb_title", "relevance"]
missing_manual = [c for c in required if c not in manual_df.columns]
if missing_manual:
    raise ValueError(f"Il file input manca delle colonne: {missing_manual}")

# Se il file manuale ha lo stesso numero di righe e lo stesso ordine, usa l'assegnazione diretta
if len(target_df) == len(manual_df):
    target_df["relevance"] = pd.to_numeric(manual_df["relevance"], errors="coerce")
    target_df.to_csv(target_path, index=False)
    print("relevance aggiornato per ordine delle righe.")
else:
    # Fallback robusto: match su query_id + tmdb_title
    merged = target_df.merge(
        manual_df[["query_id", "tmdb_title", "relevance"]].rename(
            columns={"relevance": "relevance_manual"}
        ),
        on=["query_id", "tmdb_title"],
        how="left",
    )

    # Mantiene il valore già presente nel target solo se il manuale non lo ha
    merged["relevance"] = (
        pd.to_numeric(merged["relevance_manual"], errors="coerce")
        .combine_first(pd.to_numeric(merged["relevance"], errors="coerce"))
    )

    merged = merged.drop(columns=["relevance_manual"], errors="ignore")
    merged.to_csv(target_path, index=False)
    print("relevance aggiornato con match query_id + tmdb_title.")

# Controllo finale
updated = pd.read_csv(target_path)
print("Colonne:", updated.columns.tolist())
print("relevance nulli:", int(updated["relevance"].isna().sum()))
display(updated[["query_id", "tmdb_title", "relevance"]].head(10))



relevance aggiornato con match query_id + tmdb_title.
Colonne: ['query_id', 'query', 'movieId', 'tmdb_title', 'retrieved_by', 'best_rank', 'relevance', 'notes']
relevance nulli: 0


,query_id,tmdb_title,relevance
0,Q01,Dark City,2.0
1,Q01,Memento,2.0
2,Q01,Donnie Darko,2.0
3,Q01,Eternal Sunshine of the Spotless Mind,2.0
4,Q01,The Island,1.0
5,Q01,Men in Black II,0.0
6,Q01,Identity,1.0
7,Q01,The Butterfly Effect,2.0
8,Q01,The Long Kiss Goodnight,1.0
9,Q01,Total Recall,1.0


In [50]:
deep_pool = pd.read_csv(DEEP_ANNOTATION_POOL_PATH)
deep_missing = int(deep_pool["relevance"].isna().sum())
print("Giudizi deep pool mancanti:", deep_missing)

if deep_missing == 0 and len(deep_pool) > 0:
    finalist_results = first_stage_results[first_stage_results["system"].isin(FINALIST_SYSTEMS)]
    metrics10 = evaluate_results(
        finalist_results[finalist_results["rank"] <= 10], deep_pool, k=10,
        id_column=ID_COLUMN, expected_query_ids=BENCHMARK_QUERIES["query_id"].tolist(), strict_judgments=True,
    )
    metrics20 = evaluate_results(
        finalist_results[finalist_results["rank"] <= 20], deep_pool, k=20,
        id_column=ID_COLUMN, expected_query_ids=BENCHMARK_QUERIES["query_id"].tolist(), strict_judgments=True,
    )
    metrics30 = evaluate_results(
        finalist_results[finalist_results["rank"] <= 30], deep_pool, k=30,
        id_column=ID_COLUMN, expected_query_ids=BENCHMARK_QUERIES["query_id"].tolist(), strict_judgments=True,
    )
    deep_summary = (
        metrics10.groupby("system")[["ndcg@10", "recall@10", "precision@10"]].mean()
        .join(metrics20.groupby("system")[["recall@20"]].mean())
        .join(metrics30.groupby("system")[["recall@30"]].mean())
        .sort_values("ndcg@10", ascending=False)
    )
    display(deep_summary)
else:
    print("Completare il deep pool prima della candidate-recall definitiva.")

Giudizi deep pool mancanti: 0


,ndcg@10,recall@10,precision@10,recall@20,recall@30
system,,,,,
gte_multilingual_base_dense,0.793289,0.177801,0.900,0.353503,0.522902
qwen3_0.6b_dense,0.755832,0.175669,0.890,0.346223,0.501386
gte_dense+bge_m3_sparse_rrf,0.695341,0.161421,0.820,0.303203,0.441021
bm25+bge_m3_dense_rrf,0.693804,0.174599,0.835,0.309829,0.458301


### risultati ottenuti:

| sistema | NDCG@10 | precision@10 | recall@20 | recall@30
|---------|------|--------|------|--------|
| `gte_multilingual_base_dense` |0.793289	|0.177801	|0.900	|0.353503	| 0.522902|
| `qwen3_0.6b_dense` |0.755832	|0.175669	|0.890	|0.346223	| 0.501386|
| `gte_dense+bge_m3_sparse_rrf` |0.695341	|0.161421	|0.820	|0.303203	| 0.441021|
| `bm25+bge_m3_dense_rrf` |0.693804	|0.174599	|0.835	|0.309829	| 0.458301|



## 12. Reranking dei finalisti

I reranker vengono provati solo dopo il first-stage screening/deep evaluation, evitando una griglia inutile su sistemi chiaramente inferiori.

Reranker disponibili:

- `BAAI/bge-reranker-v2-m3`;
- `Qwen/Qwen3-Reranker-0.6B`;
- `Alibaba-NLP/gte-multilingual-reranker-base`.

Ogni reranker viene provato sui candidate pool 10/20/30 di ciascun first-stage finalista e produce top-10 finali. Poiché il deep pool contiene già i top-30 dei finalisti, il reranker non dovrebbe introdurre documenti non giudicati.

In [94]:
#controllo utilizzo GPU

import torch

print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device scelto:", device)

reranker = SentenceTransformerCrossEncoderReranker(
    "Qwen/Qwen3-Reranker-0.6B",
    device="mps",
    batch_size=8,
)
reranker.load()

print(next(reranker._model.model.parameters()).device)

# se mps:0 -> allora significa che sta usando GPU Apple Silicon, se cpu -> sta usando CPU

MPS built: True
MPS available: True
Device scelto: mps
mps:0


In [ ]:
# RERANKER_SPECS_OLD = {
#     "bge_reranker_v2_m3": lambda: BGEV2M3Reranker(use_fp16=None),
#     "qwen3_reranker_0.6b": lambda: SentenceTransformerCrossEncoderReranker(
#         "Qwen/Qwen3-Reranker-0.6B", batch_size=8, max_length=1024,
#     ),
#     "gte_multilingual_reranker": lambda: SentenceTransformerCrossEncoderReranker(
#         "Alibaba-NLP/gte-multilingual-reranker-base", trust_remote_code=True, batch_size=8, max_length=1024,
#     ),
# }

def best_torch_device() -> str: #--> "mps"
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

# Tutti i reranker vedono al massimo 1024 token: confronto più controllato e tempi prevedibili.
RERANK_MAX_LENGTH = 1024
RERANK_CHECKPOINT_CHUNK_SIZE = 32
CROSS_ENCODER_MODEL_KWARGS = {"torch_dtype": torch.float16} if RERANK_DEVICE == "cuda" else {}

RERANKER_SPECS = {
    "bge_reranker_v2_m3": lambda: BGEV2M3Reranker(
        use_fp16=True if RERANK_DEVICE == "cuda" else False
    ),
    "qwen3_reranker_0.6b": lambda: SentenceTransformerCrossEncoderReranker(
        "Qwen/Qwen3-Reranker-0.6B",
        device=RERANK_DEVICE,
        batch_size=8,
        max_length=RERANK_MAX_LENGTH,
        model_kwargs=CROSS_ENCODER_MODEL_KWARGS,
    ),
    "gte_multilingual_reranker": lambda: SentenceTransformerCrossEncoderReranker(
        "Alibaba-NLP/gte-multilingual-reranker-base",
        trust_remote_code=True,
        device=RERANK_DEVICE,
        batch_size=8,
        max_length=RERANK_MAX_LENGTH,
        model_kwargs=CROSS_ENCODER_MODEL_KWARGS,
    ),
}

mps


In [90]:
# import inspect

# import importlib
# import movie_rag.experimental_retrieval as experimental_retrieval

# experimental_retrieval = importlib.reload(experimental_retrieval)

# SentenceTransformerCrossEncoderReranker = (
#     experimental_retrieval.SentenceTransformerCrossEncoderReranker
# )


# print(inspect.signature(SentenceTransformerCrossEncoderReranker))
# print(inspect.getfile(SentenceTransformerCrossEncoderReranker))



In [91]:
#TEST velocità di scoring dei reranker su 20 coppie query-documento

import time

EXCECUTE_SPEED_TEST = True

if EXCECUTE_SPEED_TEST:
    print("Esecuzione test di velocità dei reranker su 2 coppie query-documento...")
    query = BENCHMARK_QUERIES.iloc[0]["query"]
    print("Query:", query)
    docs = prepared_df[BASELINE_V2.document_column].head(20).astype(str).tolist()

    for reranker_name, factory in RERANKER_SPECS.items():
        reranker = factory()
        reranker.load()
        
        start = time.perf_counter()
        scores = reranker.score(query, docs)
        elapsed = time.perf_counter() - start

        print(f"Reranker: {reranker_name}")
        print(f"2 coppie: {elapsed:.2f} s")
        print(f"{elapsed / len(docs):.3f} s/coppia\n\n")
        # print(f"Scores: {scores}")
else:
    print("Test di velocità dei reranker saltato.")


Esecuzione test di velocità dei reranker su 2 coppie query-documento...
Query: a melancholic film about memory, identity and time


Compute Scores: 100%|██████████| 2/2 [06:38<00:00, 199.42s/it]


Reranker: bge_reranker_v2_m3
2 coppie: 817.20 s
40.860 s/coppia


Reranker: qwen3_reranker_0.6b
2 coppie: 899.30 s
44.965 s/coppia




config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/612M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker: gte_multilingual_reranker
2 coppie: 249.44 s
12.472 s/coppia




#### risultati ottenuti di stima tempi:

Reranker: bge_reranker_v2_m3\
20 coppie: 817.20 s\
40.860 s/coppia

Reranker: qwen3_reranker_0.6b\
20 coppie: 899.30 s\
44.965 s/coppia

Reranker: gte_multilingual_reranker\
20 coppie: 249.44 s\
12.472 s/coppia



In [ ]:
#calcolo coppie uniche query-documento da valutare per reranker
#dei modelli di retrieval finalisti

rerank_candidates_unique = (
    first_stage_results[
        first_stage_results["system"].isin(FINALIST_SYSTEMS)
        & first_stage_results["rank"].le(30)
    ]
    [["query_id", ID_COLUMN]]
    .drop_duplicates()
)

N = len(rerank_candidates_unique)

print("Coppie uniche da valutare per reranker:", N)



Coppie uniche da valutare per reranker: 1349


In [ ]:
## OLD

# def candidate_frame_from_results(query_id: str, system: str, pool_size: int) -> pd.DataFrame:
#     run = first_stage_results[
#         first_stage_results["query_id"].eq(query_id)
#         & first_stage_results["system"].eq(system)
#         & first_stage_results["rank"].le(pool_size)
#     ].sort_values("rank").copy()
#     positions = prepared_df.reset_index().set_index(ID_COLUMN)["index"].to_dict()
#     rows = []
#     for item in run.to_dict("records"):
#         pos = positions[item[ID_COLUMN]]
#         rows.append({
#             "rank": int(item["rank"]),
#             "score": float(item["score"]),
#             ID_COLUMN: item[ID_COLUMN],
#             TITLE_COLUMN: item[TITLE_COLUMN],
#             DOCUMENT_COLUMN: prepared_df.iloc[pos][DOCUMENT_COLUMN],
#         })
#     return pd.DataFrame(rows)


# def run_reranker_grid() -> pd.DataFrame:
#     records = []
#     for reranker_name, factory in RERANKER_SPECS.items():
#         reranker = factory()
#         reranker.load()
#         print("Reranker loaded:", reranker_name)
#         for query_row in BENCHMARK_QUERIES.to_dict("records"):
#             for first_stage_system in FINALIST_SYSTEMS:
#                 for pool_size in CANDIDATE_POOL_SIZES:
#                     candidates = candidate_frame_from_results(query_row["query_id"], first_stage_system, pool_size)
#                     if candidates.empty:
#                         continue
#                     reranked = rerank_candidates(
#                         query=query_row["query"],
#                         candidates=candidates,
#                         reranker=reranker,
#                         top_k=TOP_K,
#                         document_column=DOCUMENT_COLUMN,
#                     )
#                     system_name = f"{first_stage_system}+{reranker_name}@{pool_size}"
#                     records.extend(result_records(reranked, query_row, system_name))
#         if hasattr(reranker, "unload"):
#             reranker.unload()
#         elif hasattr(reranker, "_model"):
#             reranker._model = None
#     return pd.DataFrame(records)


# # La griglia è costosa: abilitarla solo dopo aver aggiornato FINALIST_SYSTEMS.
# RUN_RERANKER_GRID = True

# if RUN_RERANKER_GRID:
#     rerank_results = run_reranker_grid()
#     rerank_results.to_csv(RERANK_RESULTS_PATH, index=False)
#     print("Rerank rows:", len(rerank_results))
# else:
#     rerank_results = pd.read_csv(RERANK_RESULTS_PATH) if RERANK_RESULTS_PATH.exists() else pd.DataFrame()
#     print("Reranker grid non eseguita. Impostare RUN_RERANKER_GRID=True quando pronti.")

Reranker loaded: bge_reranker_v2_m3


KeyboardInterrupt: 

In [103]:
# versione NUOVA

# 1) Unione top-30 dei soli finalisti e deduplica (query_id, movieId).
unique_rerank_candidates = build_unique_rerank_candidates(
    first_stage_results=first_stage_results,
    queries=BENCHMARK_QUERIES,
    prepared_df=prepared_df,
    finalist_systems=FINALIST_SYSTEMS,
    max_pool_size=DEEP_POOL_DEPTH,
    id_column=ID_COLUMN,
    title_column=TITLE_COLUMN,
    document_column=DOCUMENT_COLUMN,
)
unique_rerank_candidates.to_csv(RERANK_CANDIDATES_PATH, index=False)

print("Coppie uniche da score per reranker:", len(unique_rerank_candidates))
print("Massimo teorico senza deduplica:", len(BENCHMARK_QUERIES) * len(FINALIST_SYSTEMS) * DEEP_POOL_DEPTH)
print("File trasferibile a Colab:", RERANK_CANDIDATES_PATH)

# 2) Scoring costoso e resumable. Lasciare False se gli score vengono calcolati su Colab.
RUN_RERANKER_SCORING = False
RERANKERS_TO_RUN = [
    "bge_reranker_v2_m3",
    "qwen3_reranker_0.6b",
    "gte_multilingual_reranker",
]

if RUN_RERANKER_SCORING:
    for reranker_name in RERANKERS_TO_RUN:
        checkpoint_path = RERANK_SCORE_DIR / f"{reranker_name}.csv"
        reranker = RERANKER_SPECS[reranker_name]()
        reranker.load()
        print("Loaded:", reranker_name)
        if isinstance(reranker, SentenceTransformerCrossEncoderReranker):
            try:
                print("Model device:", next(reranker._model.model.parameters()).device)
                print("Max sequence length:", reranker._model.max_seq_length)
            except Exception:
                pass
        score_reranker_candidates_incremental(
            candidates=unique_rerank_candidates,
            reranker=reranker,
            reranker_name=reranker_name,
            checkpoint_path=checkpoint_path,
            id_column=ID_COLUMN,
            document_column=DOCUMENT_COLUMN,
            checkpoint_chunk_size=RERANK_CHECKPOINT_CHUNK_SIZE,
            progress=True,
        )
        if hasattr(reranker, "unload"):
            reranker.unload()

# 3) Ricostruzione economica di tutti i run @10/@20/@30 dai checkpoint disponibili.
expected_keys = set(zip(
    unique_rerank_candidates["query_id"].astype(str),
    unique_rerank_candidates[ID_COLUMN].astype(str),
))
reranked_frames = []

for reranker_name in RERANKER_SPECS:
    checkpoint_path = RERANK_SCORE_DIR / f"{reranker_name}.csv"
    if not checkpoint_path.exists():
        print(f"{reranker_name}: checkpoint non presente")
        continue
    scores = pd.read_csv(checkpoint_path)
    scored_keys = set(zip(scores["query_id"].astype(str), scores[ID_COLUMN].astype(str)))
    missing = expected_keys - scored_keys
    print(f"{reranker_name}: {len(scored_keys)}/{len(expected_keys)} pair disponibili; mancanti={len(missing)}")
    if missing:
        continue
    reranked_frames.append(
        materialize_reranked_runs(
            first_stage_results=first_stage_results,
            queries=BENCHMARK_QUERIES,
            reranker_scores=scores,
            finalist_systems=FINALIST_SYSTEMS,
            candidate_pool_sizes=CANDIDATE_POOL_SIZES,
            top_k=TOP_K,
            id_column=ID_COLUMN,
            title_column=TITLE_COLUMN,
        )
    )

rerank_results = pd.concat(reranked_frames, ignore_index=True) if reranked_frames else pd.DataFrame()
if not rerank_results.empty:
    rerank_results.to_csv(RERANK_RESULTS_PATH, index=False)
    print("Rerank rows materializzate:", len(rerank_results))
else:
    print("Nessun reranker completo ancora materializzabile. Lo scoring può essere eseguito localmente o su Colab.")


Coppie uniche da score per reranker: 1349
Massimo teorico senza deduplica: 2400
File trasferibile a Colab: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/artifacts/retrieval_ablation_2000_v1/rerank_candidates_unique.csv
bge_reranker_v2_m3: 1349/1349 pair disponibili; mancanti=0
qwen3_reranker_0.6b: 1349/1349 pair disponibili; mancanti=0
gte_multilingual_reranker: 1349/1349 pair disponibili; mancanti=0
Rerank rows materializzate: 7200


## 13. Confronto finale retrieval vs reranking

La metrica principale del ranking finale rimane `NDCG@10`. Per interpretare il reranker va letta insieme alla candidate recall del first stage: se un rilevante non entra nei primi 20/30, nessun reranker può recuperarlo.

In [ ]:
if not rerank_results.empty and DEEP_ANNOTATION_POOL_PATH.exists():
    deep_pool = pd.read_csv(DEEP_ANNOTATION_POOL_PATH)
    deep_missing = int(deep_pool["relevance"].isna().sum())
    unseen = build_annotation_extension(
        rerank_results[rerank_results["rank"] <= TOP_K],
        deep_pool,
        id_column=ID_COLUMN,
        title_column=TITLE_COLUMN,
    )
    print("Giudizi deep mancanti:", deep_missing, "| documenti reranked non giudicati:", len(unseen))

    if deep_missing == 0 and unseen.empty:
        rerank_metrics = evaluate_results(
            rerank_results,
            deep_pool,
            k=TOP_K,
            id_column=ID_COLUMN,
            expected_query_ids=BENCHMARK_QUERIES["query_id"].tolist(),
            strict_judgments=True,
        )
        first_stage_final = first_stage_results[
            first_stage_results["system"].isin(FINALIST_SYSTEMS)
            & first_stage_results["rank"].le(TOP_K)
        ]
        first_stage_metrics = evaluate_results(
            first_stage_final,
            deep_pool,
            k=TOP_K,
            id_column=ID_COLUMN,
            expected_query_ids=BENCHMARK_QUERIES["query_id"].tolist(),
            strict_judgments=True,
        )
        comparison = pd.concat([first_stage_metrics, rerank_metrics], ignore_index=True)
        display(
            comparison.groupby("system")[["ndcg@10", "recall@10", "precision@10"]]
            .mean()
            .sort_values("ndcg@10", ascending=False)
        )
    else:
        print("Completare il deep pool; con candidate pool <=30 non dovrebbero servire qrels aggiuntivi.")
        
#gte_multilingual_base_dense	0.794889	0.333658	0.900
#qwen3_0.6b_dense	0.759249	0.332600	0.890

Giudizi deep mancanti: 0 | documenti reranked non giudicati: 0


,ndcg@10,recall@10,precision@10
system,,,
gte_multilingual_base_dense+gte_multilingual_reranker@20,0.809421,0.187252,0.925
gte_multilingual_base_dense+bge_reranker_v2_m3@20,0.805957,0.184122,0.930
gte_multilingual_base_dense+qwen3_reranker_0.6b@20,0.804563,0.183456,0.910
gte_multilingual_base_dense+qwen3_reranker_0.6b@30,0.802067,0.187075,0.910
gte_multilingual_base_dense+bge_reranker_v2_m3@10,0.797886,0.177801,0.900
qwen3_0.6b_dense+qwen3_reranker_0.6b@20,0.797436,0.178540,0.905
gte_dense+bge_m3_sparse_rrf+gte_multilingual_reranker@30,0.797389,0.185361,0.920
gte_multilingual_base_dense+gte_multilingual_reranker@10,0.797214,0.177801,0.900
gte_multilingual_base_dense+qwen3_reranker_0.6b@10,0.796894,0.177801,0.900


### la migliore combinazione di modelli risulta
-> gte_multilingual_base_dense+gte_multilingual_reranker@20

la recall@10 sembra piuttosto bassa, ma è da considerare che probabilmente ci sono spesso più di 10 film che rispondono alla query.\
calcoliamo la Recall massima

In [105]:
# Quanti documenti rilevanti (relevance > 0) abbiamo giudicato per ogni query?
relevant_per_query = (
    deep_pool[deep_pool["relevance"] > 0]
    .groupby("query_id")[ID_COLUMN]
    .nunique()
    .rename("relevant_judged")
    .reset_index()
)

# Massima Recall@10 matematicamente raggiungibile:
# se ho 50 film rilevanti, anche una top10 perfetta può fare solo 10/50 = 0.20
relevant_per_query["max_possible_recall@10"] = (
    np.minimum(10, relevant_per_query["relevant_judged"])
    / relevant_per_query["relevant_judged"]
)

display(relevant_per_query)

print(
    "Media film rilevanti giudicati/query:",
    relevant_per_query["relevant_judged"].mean()
)

print(
    "Recall@10 massima media possibile:",
    relevant_per_query["max_possible_recall@10"].mean()
)

,query_id,relevant_judged,max_possible_recall@10
0,Q01,59,0.169492
1,Q02,67,0.149254
2,Q03,11,0.909091
3,Q04,53,0.188679
4,Q05,42,0.238095
5,Q06,58,0.172414
6,Q07,71,0.140845
7,Q08,54,0.185185
8,Q09,32,0.312500
9,Q10,60,0.166667


Media film rilevanti giudicati/query: 50.4
Recall@10 massima media possibile: 0.23372217544905874


## 14. Decisione dopo questo notebook

Il risultato di questo esperimento **non promuove automaticamente** una nuova baseline Qdrant. La sequenza prevista è:

1. completare screening pool e confronto first-stage;
2. selezionare pochi finalisti;
3. completare il deep pool e misurare candidate Recall@20/30;
4. confrontare i reranker sui finalisti;
5. scegliere una configurazione considerando qualità, robustezza e semplicità;
6. **solo allora** promuovere rappresentazione/retriever vincente a nuova baseline Qdrant e riallineare MCP/LLM;
7. successivamente ripetere una valutazione più piccola ma domain-specific sui documenti educativi/WhoTeach.

**Limite del corpus attuale:** i ~2.000 film sono i più ricchi di rating nel subset MovieLens selezionato dal notebook 02. È una scelta utile per avere metadata relativamente completi, ma non costituisce un campione casuale dell'intero catalogo e va dichiarata come limite del proxy benchmark.

## Riferimenti implementativi

- BGE-M3: `BAAI/bge-m3` — dense + learned sparse + multi-vector.
- Qwen3 Embedding: `Qwen/Qwen3-Embedding-0.6B`.
- Qwen3 Reranker: `Qwen/Qwen3-Reranker-0.6B`.
- GTE multilingual: `Alibaba-NLP/gte-multilingual-base`.
- GTE reranker: `Alibaba-NLP/gte-multilingual-reranker-base`.

Le modalità multi-vector/ColBERT di BGE-M3 non vengono incluse in questo primo confronto per non ampliare ulteriormente la griglia: possono essere aggiunte solo se dense+sparse non risulta sufficiente.

## Valutazione tempistiche modelli di reranking:

#### Locale
| Reranker | secondi per 20 coppie | secondi a coppia
|---------|------|--------|
| `bge_reranker_v2_m3` | 817.20 s | 40.860 s/coppia |
| `qwen3_reranker_0.6b` | 899.30 s | 44.965 s/coppia | 
| `gte_multilingual_reranker` | 249.44 s | 12.472 s/coppia |

#### Colab
| Reranker | secondi per 20 coppie | secondi a coppia
|---------|------|--------|
| `bge_reranker_v2_m3` | 0.51 s | 0.026 s/coppia |
| `qwen3_reranker_0.6b` | 0.65 s | 0.032 s/coppia | 
| `gte_multilingual_reranker` | 0.16 s | 0.008 s/coppia |


# FREE QUERY 
Provare liberamente query, retriever e reranker

In [106]:
import gc
import numpy as np
import pandas as pd


AVAILABLE_RETRIEVAL_SYSTEMS = [
    "bm25",
    "bge_m3_dense",
    "bge_m3_sparse",
    "qwen3_0.6b_dense",
    "gte_multilingual_base_dense",
    "bm25+bge_m3_dense_rrf",
    "bge_m3_dense+sparse_rrf",
    "bm25+bge_m3_dense+sparse_rrf",
    "qwen3_dense+bge_m3_sparse_rrf",
    "gte_dense+bge_m3_sparse_rrf",
]

AVAILABLE_RERANKERS = [
    None,
    "bge_reranker_v2_m3",
    "qwen3_reranker_0.6b",
    "gte_multilingual_reranker",
]


def retrieve_free_query(
    query: str,
    retrieval_system: str,
    *,
    filters: dict | None = None,
    depth: int = 30,
) -> pd.DataFrame:
    """
    Retrieval di una query arbitraria usando gli embedding documentali
    già calcolati nel notebook.

    Viene calcolato soltanto l'embedding della nuova query.
    """
    if retrieval_system not in AVAILABLE_RETRIEVAL_SYSTEMS:
        raise ValueError(
            f"Retriever sconosciuto: {retrieval_system}\n"
            f"Disponibili: {AVAILABLE_RETRIEVAL_SYSTEMS}"
        )

    filters = filters or {}

    # Determiniamo quali componenti servono.
    needs_bm25 = retrieval_system in {
        "bm25",
        "bm25+bge_m3_dense_rrf",
        "bm25+bge_m3_dense+sparse_rrf",
    }

    needs_bge_dense = retrieval_system in {
        "bge_m3_dense",
        "bm25+bge_m3_dense_rrf",
        "bge_m3_dense+sparse_rrf",
        "bm25+bge_m3_dense+sparse_rrf",
    }

    needs_bge_sparse = retrieval_system in {
        "bge_m3_sparse",
        "bge_m3_dense+sparse_rrf",
        "bm25+bge_m3_dense+sparse_rrf",
        "qwen3_dense+bge_m3_sparse_rrf",
        "gte_dense+bge_m3_sparse_rrf",
    }

    needs_qwen = retrieval_system in {
        "qwen3_0.6b_dense",
        "qwen3_dense+bge_m3_sparse_rrf",
    }

    needs_gte = retrieval_system in {
        "gte_multilingual_base_dense",
        "gte_dense+bge_m3_sparse_rrf",
    }

    frames = {}

    # ----------------------------------------------------------
    # BM25
    # ----------------------------------------------------------
    if needs_bm25:
        frames["bm25"] = retrieve_from_scores(
            prepared_df,
            BM25.scores(query),
            depth,
            filters,
            ID_COLUMN,
            TITLE_COLUMN,
            DOCUMENT_COLUMN,
        )

    # ----------------------------------------------------------
    # BGE-M3 dense / sparse
    # Una sola codifica della query produce entrambe.
    # ----------------------------------------------------------
    if needs_bge_dense or needs_bge_sparse:
        encoder = BGEM3MultiEncoder()

        try:
            dense_q, sparse_q = encoder.encode_query(query)

            if needs_bge_dense:
                frames["bge_m3_dense"] = retrieve_from_scores(
                    prepared_df,
                    dense_scores(dense_q, bge_dense_vectors),
                    depth,
                    filters,
                    ID_COLUMN,
                    TITLE_COLUMN,
                    DOCUMENT_COLUMN,
                )

            if needs_bge_sparse:
                frames["bge_m3_sparse"] = retrieve_from_scores(
                    prepared_df,
                    sparse_scores(sparse_q, bge_sparse_documents),
                    depth,
                    filters,
                    ID_COLUMN,
                    TITLE_COLUMN,
                    DOCUMENT_COLUMN,
                )
        finally:
            encoder.unload()

    # ----------------------------------------------------------
    # Qwen3 dense
    # ----------------------------------------------------------
    if needs_qwen:
        encoder = SentenceTransformerDenseEncoder(QWEN_SPEC)

        try:
            q = encoder.encode_query(query)

            frames["qwen3_0.6b_dense"] = retrieve_from_scores(
                prepared_df,
                dense_scores(q, qwen_dense_vectors),
                depth,
                filters,
                ID_COLUMN,
                TITLE_COLUMN,
                DOCUMENT_COLUMN,
            )
        finally:
            encoder.unload()

    # ----------------------------------------------------------
    # GTE dense
    # ----------------------------------------------------------
    if needs_gte:
        encoder = SentenceTransformerDenseEncoder(GTE_SPEC)

        try:
            q = encoder.encode_query(query)

            frames["gte_multilingual_base_dense"] = retrieve_from_scores(
                prepared_df,
                dense_scores(q, gte_dense_vectors),
                depth,
                filters,
                ID_COLUMN,
                TITLE_COLUMN,
                DOCUMENT_COLUMN,
            )
        finally:
            encoder.unload()

    # ----------------------------------------------------------
    # Sistemi singoli
    # ----------------------------------------------------------
    if retrieval_system in frames:
        return frames[retrieval_system].copy()

    # ----------------------------------------------------------
    # Hybrid RRF
    # ----------------------------------------------------------
    if retrieval_system == "bm25+bge_m3_dense_rrf":
        return weighted_rrf(
            {
                "bm25": frames["bm25"],
                "bge_m3_dense": frames["bge_m3_dense"],
            },
            weights={
                "bm25": 0.5,
                "bge_m3_dense": 0.5,
            },
            rrf_k=RRF_K,
            top_k=depth,
            id_column=ID_COLUMN,
        )

    if retrieval_system == "bge_m3_dense+sparse_rrf":
        return weighted_rrf(
            {
                "bge_m3_dense": frames["bge_m3_dense"],
                "bge_m3_sparse": frames["bge_m3_sparse"],
            },
            weights={
                "bge_m3_dense": 0.5,
                "bge_m3_sparse": 0.5,
            },
            rrf_k=RRF_K,
            top_k=depth,
            id_column=ID_COLUMN,
        )

    if retrieval_system == "bm25+bge_m3_dense+sparse_rrf":
        return weighted_rrf(
            {
                "bm25": frames["bm25"],
                "bge_m3_dense": frames["bge_m3_dense"],
                "bge_m3_sparse": frames["bge_m3_sparse"],
            },
            weights={
                "bm25": 1 / 3,
                "bge_m3_dense": 1 / 3,
                "bge_m3_sparse": 1 / 3,
            },
            rrf_k=RRF_K,
            top_k=depth,
            id_column=ID_COLUMN,
        )

    if retrieval_system == "qwen3_dense+bge_m3_sparse_rrf":
        return weighted_rrf(
            {
                "qwen3_0.6b_dense": frames["qwen3_0.6b_dense"],
                "bge_m3_sparse": frames["bge_m3_sparse"],
            },
            weights={
                "qwen3_0.6b_dense": 0.5,
                "bge_m3_sparse": 0.5,
            },
            rrf_k=RRF_K,
            top_k=depth,
            id_column=ID_COLUMN,
        )

    if retrieval_system == "gte_dense+bge_m3_sparse_rrf":
        return weighted_rrf(
            {
                "gte_multilingual_base_dense":
                    frames["gte_multilingual_base_dense"],
                "bge_m3_sparse":
                    frames["bge_m3_sparse"],
            },
            weights={
                "gte_multilingual_base_dense": 0.5,
                "bge_m3_sparse": 0.5,
            },
            rrf_k=RRF_K,
            top_k=depth,
            id_column=ID_COLUMN,
        )

    raise RuntimeError("Sistema non gestito")


def search_movies(
    query: str,
    retrieval_system: str = "gte_multilingual_base_dense",
    reranker_name: str | None = None,
    *,
    candidate_pool_size: int = 20,
    top_k: int = 10,
    filters: dict | None = None,
) -> pd.DataFrame:
    """
    Query completa:

        query
          -> retriever
          -> top candidate_pool_size
          -> eventuale reranker
          -> top_k

    reranker_name=None significa nessun reranking.
    """

    if top_k > candidate_pool_size:
        raise ValueError("top_k deve essere <= candidate_pool_size")

    candidates = retrieve_free_query(
        query=query,
        retrieval_system=retrieval_system,
        filters=filters,
        depth=candidate_pool_size,
    )

    # Senza reranker
    if reranker_name is None:
        result = candidates.head(top_k).copy()
        result["retrieval_system"] = retrieval_system
        result["reranker"] = None

        return result

    if reranker_name not in RERANKER_SPECS:
        raise ValueError(
            f"Reranker sconosciuto: {reranker_name}\n"
            f"Disponibili: {list(RERANKER_SPECS)}"
        )

    reranker = RERANKER_SPECS[reranker_name]()

    try:
        reranker.load()

        result = rerank_candidates(
            query=query,
            candidates=candidates.head(candidate_pool_size),
            reranker=reranker,
            top_k=top_k,
            document_column=DOCUMENT_COLUMN,
        )
    finally:
        if hasattr(reranker, "unload"):
            reranker.unload()
        del reranker
        gc.collect()

    result["retrieval_system"] = retrieval_system
    result["reranker"] = reranker_name

    return result

In [107]:
QUERY = "a dark science fiction film about surveillance and social control"

RETRIEVAL = "gte_multilingual_base_dense"

RERANKER = "gte_multilingual_reranker"
# oppure:
# RERANKER = "bge_reranker_v2_m3"
# RERANKER = "qwen3_reranker_0.6b"
# RERANKER = None

results = search_movies(
    query=QUERY,
    retrieval_system=RETRIEVAL,
    reranker_name=RERANKER,
    candidate_pool_size=20,
    top_k=10,
    filters={},
)

display(
    results[
        [
            "rank",
            TITLE_COLUMN,
            "score",
            "retrieval_system",
            "reranker",
        ]
    ]
)

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


,rank,tmdb_title,score,retrieval_system,reranker
0,1,Minority Report,0.605992,gte_multilingual_base_dense,gte_multilingual_reranker
1,2,Elysium,0.568228,gte_multilingual_base_dense,gte_multilingual_reranker
2,3,District 9,0.563493,gte_multilingual_base_dense,gte_multilingual_reranker
3,4,A Clockwork Orange,0.551224,gte_multilingual_base_dense,gte_multilingual_reranker
4,5,Enemy of the State,0.548670,gte_multilingual_base_dense,gte_multilingual_reranker
5,6,The Conversation,0.515749,gte_multilingual_base_dense,gte_multilingual_reranker
6,7,Nineteen Eighty-Four,0.508321,gte_multilingual_base_dense,gte_multilingual_reranker
7,8,A Scanner Darkly,0.500447,gte_multilingual_base_dense,gte_multilingual_reranker
8,9,Strange Days,0.490991,gte_multilingual_base_dense,gte_multilingual_reranker
9,10,V for Vendetta,0.461763,gte_multilingual_base_dense,gte_multilingual_reranker


In [108]:
QUERY = "a dark science fiction film about surveillance and social control"

RETRIEVAL = "gte_multilingual_base_dense"

RERANKER = "gte_multilingual_reranker"
# oppure:
# RERANKER = "bge_reranker_v2_m3"
# RERANKER = "qwen3_reranker_0.6b"
# RERANKER = None

results = search_movies(
    query=QUERY,
    retrieval_system=RETRIEVAL,
    reranker_name=None,
    candidate_pool_size=20,
    top_k=10,
    filters={},
)

display(
    results[
        [
            "rank",
            TITLE_COLUMN,
            "score",
            "retrieval_system",
            "reranker",
        ]
    ]
)

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


,rank,tmdb_title,score,retrieval_system,reranker
0,1,Enemy of the State,0.678637,gte_multilingual_base_dense,None
1,2,V for Vendetta,0.670112,gte_multilingual_base_dense,None
2,3,Minority Report,0.667003,gte_multilingual_base_dense,None
3,4,A Clockwork Orange,0.662945,gte_multilingual_base_dense,None
4,5,Surrogates,0.660660,gte_multilingual_base_dense,None
5,6,Strange Days,0.659911,gte_multilingual_base_dense,None
6,7,Nineteen Eighty-Four,0.659784,gte_multilingual_base_dense,None
7,8,District 9,0.653558,gte_multilingual_base_dense,None
8,9,The Conversation,0.650324,gte_multilingual_base_dense,None
9,10,Dark City,0.647563,gte_multilingual_base_dense,None


In [ ]:
results = search_movies(
    query="a thoughtful film about artificial intelligence and human identity",
    retrieval_system="gte_multilingual_base_dense",
    reranker_name="gte_multilingual_reranker",
    candidate_pool_size=20,
    top_k=10,
    filters={
        "min_year": 2000,
        "include_genres": ["Science Fiction"],
    },
)